# CMIP6 IITM-ESM daily precipitation dataset-source path error

When: 2026-10-25

The reported request subsets daily precipitation (`pr`) from IITM-ESM SSP2-4.5, realization `r1i1p1f1`, version `v20200915`, over area `113.0,1.5,130.5,24.0` for 2015–2100. It selects every day and month through `time_components` as well as `time="2015/2100"`.

The service reported: `Process error: A dataset source requires at least one path.`

This notebook preserves the supplied status and request and builds the equivalent workflow with Rooki's Python operators. The reproduction has not been run here; code cells have no recorded outputs.


## Reported WPS status

```xml
<wps:Status creationTime="2026-10-25T18:19:46Z">
    <wps:ProcessFailed>
        <wps:ExceptionReport>
            <ows:Exception exceptionCode="NoApplicableCode" locator="None">
                <ows:ExceptionText>Process error: A dataset source requires at least one path.</ows:ExceptionText>
            </ows:Exception>
        </wps:ExceptionReport>
    </wps:ProcessFailed>
</wps:Status>
```


## Original WPS workflow

The decoded payload retains the exact collection, area, temporal filters, and step references. Day and year lists are assembled programmatically for readability, preserving zero-padded days and the inclusive 2015–2100 range.


In [ ]:
days = ",".join(f"{day:02d}" for day in range(1, 32))
months = "jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec"
years = ",".join(str(year) for year in range(2015, 2101))
time_components = f"day:{days}|month:{months}|year:{years}"

request = {
    "inputs": {
        "pr": [
            "c3s-cmip6.ScenarioMIP.CCCR-IITM.IITM-ESM."
            "ssp245.r1i1p1f1.day.pr.gn.v20200915"
        ]
    },
    "steps": {
        "subset_pr_1": {
            "run": "subset",
            "in": {
                "collection": "inputs/pr",
                "area": "113.0,1.5,130.5,24.0",
                "time_components": time_components,
                "time": "2015/2100",
            },
        }
    },
    "outputs": {"output": "subset_pr_1/output"},
    "doc": "workflow",
}

request


## Build the equivalent Rooki workflow

Importing Rooki contacts the configured WPS service. Change `ROOK_URL` to test another deployment. The serialization assertion checks that the Python workflow matches the original request.


In [ ]:
import json
import os

os.environ["ROOK_URL"] = "http://rook.dkrz.de/wps"

from rooki import operators as ops


In [ ]:
pr = ops.Input("pr", request["inputs"]["pr"])
parameters = request["steps"]["subset_pr_1"]["in"]
workflow = ops.Subset(
    pr,
    area=parameters["area"],
    time_components=parameters["time_components"],
    time=parameters["time"],
)

serialized_request = json.loads(workflow._serialise())
assert serialized_request == request
serialized_request


## Reproduce and inspect the response

Set `RUN_REQUEST = True` to submit the full 86-year daily request to the selected service. The cell displays the response and status and lists successful output URLs without downloading result data.


In [ ]:
from time import perf_counter
from IPython.display import display

RUN_REQUEST = False

if RUN_REQUEST:
    started_at = perf_counter()
    response = workflow.orchestrate()
    print(f"Orchestration time: {perf_counter() - started_at:.1f} seconds")
    print(f"Succeeded: {response.ok}")
    print(f"Status: {response.status}")
    display(response)
    if response.ok:
        for url in response.download_urls():
            print(url)
else:
    print("Request not submitted; set RUN_REQUEST = True to reproduce it.")


## Investigation notes

The same error was reported in [the 2026-08-26 IITM-ESM notebook](cds-error-2026-08-26-cmip6-iitm-pr-mixed-frequency.ipynb), which combines monthly and daily precipitation collections. This request contains only the daily collection and uses a different area and an explicit day filter, so mixing frequencies is not required for the reported failure.

The message indicates that a dataset source was constructed without any paths. It does not establish why the paths were empty. Inspect the resolved files and catalog mapping for the exact daily collection and version, check their temporal coverage against the requested filters, and capture the server traceback before attributing the failure to a specific cause.
